In [98]:
import numpy as np
import json
import requests
from pathlib import Path

url = "http://localhost:8888/infer"
payload_path = Path("/home/tsuruoka/hdd/BEV/CarlaRunner/DrivingAgent/sample_payload.json")

with payload_path.open() as f:
    payload = json.load(f)

response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()

data = response.json()
print(data["predictions"][0].keys())




dict_keys(['SC_metric', 'SSC_metric', 'pred_c', 'pred_f'])


In [100]:
import torch
import torch.nn.functional as F
import numpy as np

# OpenOccupancy の pred_c: (C, D, H, W) = (17, 10, 128, 128)
raw_pred = np.array(data["predictions"][0]["pred_f"][0], dtype=np.float32)
print("raw pred shape:", raw_pred.shape)

# PyTorch の interpolate が期待する (N, C, D, H, W) へ拡張
tensor = torch.from_numpy(raw_pred).unsqueeze(0)
upsampled = F.interpolate(tensor, size=(40, 512, 512), mode="trilinear", align_corners=False)[0]

# クラス次元 (dim=0) で argmax -> (D, H, W)
logits = torch.argmax(upsampled, dim=0)
# (H, W, D) = (512, 512, 40) に並べ替えて numpy.int16 へ
arr = logits.permute(1, 2, 0).contiguous().cpu().numpy().astype(np.int16)

unique, counts = np.unique(arr, return_counts=True)
dist = dict(zip(unique.tolist(), counts.tolist()))
print("class histogram:", dist)

np.save("pred_c.npy", arr)


TypeError: 'NoneType' object is not subscriptable

In [95]:
import numpy as np
import torch
import torch.nn.functional as F

logits = torch.from_numpy(np.load("/home/tsuruoka/hdd/BEV/OpenOccupancy/pred.npy"))         # (1,17,128,128,10)
logits = logits.permute(0,1,4,2,3)                        # -> (1,17,10,128,128)
upsampled = F.interpolate(logits, size=(40,512,512), mode="trilinear", align_corners=False)
labels = torch.argmax(upsampled[0], dim=0)                # (10,512,512)
grid = labels.permute(1,2,0).contiguous().cpu().numpy().astype(np.int16)  # (512,512,40)
np.save("pred_c.npy", grid)


In [96]:
import sys
sys.path.append("/home/tsuruoka/hdd/BEV/OpenOccupancy")

from src.utils.occupancy_visualizer import OccupancyGridVisualizer



visualizer = OccupancyGridVisualizer(voxel_size=1.0, z_scale=1.0, default_mode="scatter")
pred_file = "/home/tsuruoka/hdd/BEV/CarlaRunner/DrivingAgent/notebooks/pred_c.npy"
#pred_file = "/home/tsuruoka/hdd/BEV/OpenOccupancy/pred.npy"
visualizer.visualize_file("/home/tsuruoka/hdd/BEV/CarlaRunner/DrivingAgent/notebooks/pred_c.npy", "Pseudo GT - Voxel Points")

grid shape: (512, 512, 40), dtype: int16


ValueError: データが空です